In [1]:
import os
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

spark = SparkSession.builder \
    .appName("GarbageStats") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 14:03:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Raw DATA

In [2]:
df_train = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/train")

df_test = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/test")

In [3]:
from pyspark.sql.functions import lit, element_at, size, count, when, col, min, max, avg, split

In [4]:
df_train = df_train.withColumn("split", lit("train"))
df_test = df_test.withColumn("split", lit("test"))

df_raw = df_train.unionByName(df_test)

df_raw = df_raw.withColumn(
    "category",
    element_at(split("path", "/"), -2)
)

In [5]:
df_raw.count()

10464

In [6]:
df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

+----+----------------+------+-------+-----+--------+
|path|modificationTime|length|content|split|category|
+----+----------------+------+-------+-----+--------+
|   0|               0|     0|      0|    0|       0|
+----+----------------+------+-------+-----+--------+



In [7]:
df_raw.groupBy("split").count().show()

+-----+-----+
|split|count|
+-----+-----+
|train| 7324|
| test| 3140|
+-----+-----+



In [8]:
df_raw.groupBy("category").count().orderBy("count", ascending=False).show()

+-------------+-----+
|     category|count|
+-------------+-----+
|        glass| 2649|
|biodegradable| 2220|
|        paper| 1746|
|    cardboard| 1439|
|        metal| 1248|
|      plastic| 1162|
+-------------+-----+



In [9]:
df_raw.select("length").describe().show()

+-------+------------------+
|summary|            length|
+-------+------------------+
|  count|             10464|
|   mean|19151.321769877675|
| stddev| 9474.638532461628|
|    min|              4188|
|    max|             67006|
+-------+------------------+



In [10]:
df_raw.groupBy("path").count().filter("count > 1").show() #doublons

+----+-----+
|path|count|
+----+-----+
+----+-----+



# DATA Transformées

In [11]:
df_train = spark.read.parquet("../data/train_data.parquet")
df_test = spark.read.parquet("../data/test_data.parquet")

In [12]:
train_count = df_train.count()
test_count = df_test.count()

total = train_count + test_count

print("Train :", train_count)
print("Test :", test_count)
print("Total :", total)

Train : 7324
Test : 3140
Total : 10464


In [13]:
df_train.printSchema()
df_test.printSchema()

root
 |-- x_train: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_train: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)

root
 |-- x_test: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_test: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)



In [14]:
df_train.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_train.columns
]).show()

+-------+-------+-----+
|x_train|y_train|class|
+-------+-------+-----+
|      0|      0|    0|
+-------+-------+-----+



In [15]:
df_test.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_test.columns
]).show()

+------+------+-----+
|x_test|y_test|class|
+------+------+-----+
|     0|     0|    0|
+------+------+-----+



In [16]:
df_train.groupBy("class").count().orderBy("count", ascending=False).show()
df_test.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass| 1854|
|biodegradable| 1554|
|        paper| 1223|
|    cardboard| 1006|
|        metal|  874|
|      plastic|  813|
+-------------+-----+

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass|  795|
|biodegradable|  666|
|        paper|  523|
|    cardboard|  433|
|        metal|  374|
|      plastic|  349|
+-------------+-----+



In [17]:
df_train = df_train.withColumn("height", size("x_train"))
df_train = df_train.withColumn("width", size(col("x_train")[0]))

df_train.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 7324|
+------+-----+-----+



In [18]:
df_test = df_test.withColumn("height", size("x_test"))
df_test = df_test.withColumn("width", size(col("x_test")[0]))

df_test.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 3140|
+------+-----+-----+



In [19]:
df_train.withColumn("label_size", size("y_train")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 7324|
+----------+-----+



In [20]:
df_test.withColumn("label_size", size("y_test")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 3140|
+----------+-----+



# Prédiction DATA

In [21]:
df_pred = spark.read.parquet("../data/prediction_data.parquet")

In [22]:
 df_pred.count()

15

In [23]:
df_pred.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|      plastic|    5|
|    cardboard|    3|
|        paper|    3|
|        metal|    2|
|biodegradable|    1|
|        glass|    1|
+-------------+-----+



In [24]:
df_pred.groupBy("file").count().filter("count > 1").show()

+----+-----+
|file|count|
+----+-----+
+----+-----+



In [25]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).show()

+----+----------+--------+-----+----------+
|file|prediction|class_id|class|confidence|
+----+----------+--------+-----+----------+
|   0|         0|       0|    0|         0|
+----+----------+--------+-----+----------+



In [26]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .show()

+-------------+-----+------------------+
|        class|count|        percentage|
+-------------+-----+------------------+
|      plastic|    5| 33.33333333333333|
|    cardboard|    3|              20.0|
|        paper|    3|              20.0|
|        metal|    2|13.333333333333334|
|biodegradable|    1| 6.666666666666667|
|        glass|    1| 6.666666666666667|
+-------------+-----+------------------+



In [27]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .show()

+-------------+-------------------+
|        class|     avg_confidence|
+-------------+-------------------+
|    cardboard| 0.7872063120206197|
|      plastic| 0.6165639221668243|
|        glass|0.45759397745132446|
|biodegradable| 0.3927757740020752|
|        metal| 0.3690199553966522|
|        paper| 0.3580915282169978|
+-------------+-------------------+



In [28]:
df_pred.filter(col("confidence") < 0.5).show()

+------------------+--------------------+--------+-------------+-------------------+
|              file|          prediction|class_id|        class|         confidence|
+------------------+--------------------+--------+-------------+-------------------+
|        glass7.jpg|[0.00376888969913...|       6|      plastic|0.46028026938438416|
|biodegradable8.jpg|[0.39277577400207...|       1|biodegradable| 0.3927757740020752|
|        glass8.jpg|[0.03651176765561...|       6|      plastic| 0.3136882185935974|
|biodegradable9.jpg|[0.06108617410063...|       3|        glass|0.45759397745132446|
|    cardboard2.jpg|[0.11384712159633...|       5|        paper|  0.374689519405365|
|      plastic5.jpg|[0.15577718615531...|       5|        paper|0.21764616668224335|
|        metal5.jpg|[0.00615987414494...|       4|        metal| 0.4887853264808655|
|        paper4.jpg|[0.00948059745132...|       5|        paper|  0.481938898563385|
|        metal4.jpg|[0.23270098865032...|       4|        metal|0

In [29]:
df_pred.select("confidence").describe().show()

+-------+-------------------+
|summary|         confidence|
+-------+-------------------+
|  count|                 15|
|   mean| 0.5404748529195785|
| stddev|0.23384530526993863|
|    min|0.21764616668224335|
|    max| 0.9976165294647217|
+-------+-------------------+



In [30]:
df_pred.orderBy("confidence", ascending=False).show(10)

+------------------+--------------------+--------+---------+-------------------+
|              file|          prediction|class_id|    class|         confidence|
+------------------+--------------------+--------+---------+-------------------+
|    cardboard3.jpg|[7.14390380185392...|       2|cardboard| 0.9976165294647217|
|    cardboard1.jpg|[9.02833107829792...|       2|cardboard| 0.8375691175460815|
|      plastic7.jpg|[1.73855703906156...|       6|  plastic| 0.7914822697639465|
|      plastic6.jpg|[0.00162648921832...|       6|  plastic| 0.7778714299201965|
|        glass9.jpg|[1.07572725482896...|       6|  plastic| 0.7394974231719971|
|        paper3.jpg|[0.07228903472423...|       2|cardboard| 0.5264332890510559|
|        metal5.jpg|[0.00615987414494...|       4|    metal| 0.4887853264808655|
|        paper4.jpg|[0.00948059745132...|       5|    paper|  0.481938898563385|
|        glass7.jpg|[0.00376888969913...|       6|  plastic|0.46028026938438416|
|biodegradable9.jpg|[0.06108

# Ecriture parquet 

In [31]:
df_pred.select(count("*").alias("total")) \
    .write.mode("overwrite") \
    .parquet("../data/stats/total.parquet")

In [32]:
df_pred.groupBy("class") \
    .count() \
    .orderBy("count", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/count_by_class.parquet")

In [33]:
df_pred.groupBy("file") \
    .count() \
    .filter("count > 1") \
    .write.mode("overwrite") \
    .parquet("../data/stats/duplicates.parquet")

In [34]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).write.mode("overwrite") \
 .parquet("../data/stats/nulls.parquet")

In [35]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/percentage_by_class.parquet")

In [36]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_by_class.parquet")

In [37]:
df_pred.filter(col("confidence") < 0.5) \
    .write.mode("overwrite") \
    .parquet("../data/stats/low_confidence.parquet")

In [38]:
df_pred.select("confidence") \
    .describe() \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_stats.parquet")

In [39]:
df_pred.orderBy("confidence", ascending=False) \
    .limit(10) \
    .write.mode("overwrite") \
    .parquet("../data/stats/top_predictions.parquet")